# VIS Classical vs 10-Band V7 + CenterNet

This notebook compares one tile in the **VIS frame** only:
- **Classical VIS** source finding at native Euclid VIS resolution
- **Neural 10-band** detections from V7 + CenterNet, also shown on VIS

The neural detector still uses all 10 bands as input. We just visualize and score both methods on the VIS image so the comparison is easy to read.


In [ ]:
import sys
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.spatial import cKDTree


def find_repo_root(start: Optional[Path] = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for cand in [start, *start.parents]:
        if (cand / 'models').exists() and (cand / 'data').exists():
            return cand
    raise FileNotFoundError('Could not find repo root containing models/ and data/.')


REPO = find_repo_root()
MODELS = REPO / 'models'
for path in [MODELS, MODELS / 'detection', MODELS / 'astrometry2']:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from jaisp_foundation_v7 import JAISPFoundationV7
from detection.centernet_detector import CenterNetDetector
from detection.detector import JAISPEncoderWrapper
from detection.dataset import _pseudo_labels_vis

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Repo root: {REPO}')
print(f'Device: {DEVICE}')


def safe_rms_from_var(var: np.ndarray) -> np.ndarray:
    var = np.asarray(var, dtype=np.float32)
    good = np.isfinite(var) & (var > 0) & (var < 1e20)
    fallback = float(np.sqrt(np.median(var[good]))) if good.any() else 1.0
    out = np.full(var.shape, fallback, dtype=np.float32)
    out[good] = np.sqrt(var[good])
    return out


def to_tensor_1chw(arr: np.ndarray, device: torch.device) -> torch.Tensor:
    arr = np.asarray(arr, dtype=np.float32)
    return torch.from_numpy(arr[None, None]).to(device)


def vis_centroids_to_xy(centroids_norm: np.ndarray, hw: tuple[int, int]) -> np.ndarray:
    h, w = hw
    if len(centroids_norm) == 0:
        return np.zeros((0, 2), dtype=np.float32)
    return np.stack([
        centroids_norm[:, 0] * max(w - 1, 1),
        centroids_norm[:, 1] * max(h - 1, 1),
    ], axis=1).astype(np.float32)


def compare_detection_sets(pred_xy: np.ndarray, ref_xy: np.ndarray, radius_px: float = 5.0) -> dict:
    pred_xy = np.asarray(pred_xy, dtype=np.float32)
    ref_xy = np.asarray(ref_xy, dtype=np.float32)
    out = {
        'matched_pred_mask': np.zeros(len(pred_xy), dtype=bool),
        'matched_ref_mask': np.zeros(len(ref_xy), dtype=bool),
        'matched_pred_count': 0,
        'matched_ref_count': 0,
    }
    if len(pred_xy) == 0 or len(ref_xy) == 0:
        return out

    tree = cKDTree(ref_xy)
    dist, idx = tree.query(pred_xy, distance_upper_bound=radius_px)
    matched_pred = np.isfinite(dist)
    matched_ref = np.zeros(len(ref_xy), dtype=bool)
    if matched_pred.any():
        matched_ref[np.unique(idx[matched_pred])] = True

    out['matched_pred_mask'] = matched_pred
    out['matched_ref_mask'] = matched_ref
    out['matched_pred_count'] = int(matched_pred.sum())
    out['matched_ref_count'] = int(matched_ref.sum())
    return out


def stretch_image(img: np.ndarray, lo: float = 1.0, hi: float = 99.5) -> np.ndarray:
    img = np.asarray(img, dtype=np.float32)
    finite = np.isfinite(img)
    if not finite.any():
        return np.zeros_like(img, dtype=np.float32)
    vals = img[finite]
    vmin, vmax = np.percentile(vals, [lo, hi])
    if vmax <= vmin:
        vmax = vmin + 1e-6
    return np.clip((np.nan_to_num(img, nan=vmin) - vmin) / (vmax - vmin), 0.0, 1.0)


def load_foundation_encoder(ckpt_path: Path, device: torch.device):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    cfg = ckpt.get('config', {})
    foundation = JAISPFoundationV7(
        band_names=cfg.get('band_names'),
        stem_ch=cfg.get('stem_ch', 64),
        hidden_ch=cfg.get('hidden_ch', 256),
        blocks_per_stage=cfg.get('blocks_per_stage', 2),
        transformer_depth=cfg.get('transformer_depth', 4),
        transformer_heads=cfg.get('transformer_heads', 8),
        fused_pixel_scale_arcsec=cfg.get('fused_pixel_scale_arcsec', 0.8),
    )
    foundation.load_state_dict(ckpt['model'], strict=False)
    foundation = foundation.to(device).eval()
    encoder = JAISPEncoderWrapper(foundation, freeze=True).to(device)
    return foundation, encoder


def load_tile_pair(stem: str, rubin_dir: Path, euclid_dir: Path, device: torch.device) -> dict:
    rubin_path = rubin_dir / f'{stem}.npz'
    euclid_path = euclid_dir / f'{stem}_euclid.npz'
    if not rubin_path.exists():
        raise FileNotFoundError(rubin_path)
    if not euclid_path.exists():
        raise FileNotFoundError(euclid_path)

    r = np.load(rubin_path, allow_pickle=True)
    e = np.load(euclid_path, allow_pickle=True)

    rubin_img = np.nan_to_num(np.asarray(r['img'], dtype=np.float32), nan=0.0)
    rubin_var = np.asarray(r['var'], dtype=np.float32)
    rubin_rms = np.stack([safe_rms_from_var(v) for v in rubin_var], axis=0)

    vis_img = np.nan_to_num(np.asarray(e['img_VIS'], dtype=np.float32), nan=0.0)
    vis_hw = tuple(vis_img.shape)

    images = {
        'rubin_u': to_tensor_1chw(rubin_img[0], device),
        'rubin_g': to_tensor_1chw(rubin_img[1], device),
        'rubin_r': to_tensor_1chw(rubin_img[2], device),
        'rubin_i': to_tensor_1chw(rubin_img[3], device),
        'rubin_z': to_tensor_1chw(rubin_img[4], device),
        'rubin_y': to_tensor_1chw(rubin_img[5], device),
        'euclid_VIS': to_tensor_1chw(np.nan_to_num(np.asarray(e['img_VIS'], dtype=np.float32), nan=0.0), device),
        'euclid_Y': to_tensor_1chw(np.nan_to_num(np.asarray(e['img_Y'], dtype=np.float32), nan=0.0), device),
        'euclid_J': to_tensor_1chw(np.nan_to_num(np.asarray(e['img_J'], dtype=np.float32), nan=0.0), device),
        'euclid_H': to_tensor_1chw(np.nan_to_num(np.asarray(e['img_H'], dtype=np.float32), nan=0.0), device),
    }
    rms = {
        'rubin_u': to_tensor_1chw(rubin_rms[0], device),
        'rubin_g': to_tensor_1chw(rubin_rms[1], device),
        'rubin_r': to_tensor_1chw(rubin_rms[2], device),
        'rubin_i': to_tensor_1chw(rubin_rms[3], device),
        'rubin_z': to_tensor_1chw(rubin_rms[4], device),
        'rubin_y': to_tensor_1chw(rubin_rms[5], device),
        'euclid_VIS': to_tensor_1chw(safe_rms_from_var(e['var_VIS']), device),
        'euclid_Y': to_tensor_1chw(safe_rms_from_var(e['var_Y']), device),
        'euclid_J': to_tensor_1chw(safe_rms_from_var(e['var_J']), device),
        'euclid_H': to_tensor_1chw(safe_rms_from_var(e['var_H']), device),
    }

    return {
        'stem': stem,
        'rubin_path': rubin_path,
        'euclid_path': euclid_path,
        'vis_img': vis_img,
        'vis_hw': vis_hw,
        'images': images,
        'rms': rms,
    }


In [ ]:
RUBIN_DIR = REPO / 'data' / 'rubin_tiles_patch25_box16'
EUCLID_DIR = REPO / 'data' / 'euclid_tiles_patch25_box16'
FOUNDATION_CKPT = REPO / 'checkpoints' / 'jaisp_v7_tiles_all_ddp_online' / 'checkpoint_best.pt'
DETECTOR_CKPT = REPO / 'checkpoints' / 'centernet_v7_patch25_box16_round1' / 'centernet_round1.pt'

TILE_NAME = 'tile_x00512_y00000_tract5063_patch_25'
CLASSICAL_VIS_NSIG = 3.0
NEURAL_CONF_THRESHOLD = 0.30
MATCH_RADIUS_VIS_PX = 5.0

assert RUBIN_DIR.exists(), RUBIN_DIR
assert EUCLID_DIR.exists(), EUCLID_DIR
assert FOUNDATION_CKPT.exists(), FOUNDATION_CKPT
assert DETECTOR_CKPT.exists(), DETECTOR_CKPT
print(f'Comparing tile: {TILE_NAME}')


In [ ]:
tile = load_tile_pair(TILE_NAME, RUBIN_DIR, EUCLID_DIR, DEVICE)
foundation, encoder = load_foundation_encoder(FOUNDATION_CKPT, DEVICE)
detector = CenterNetDetector.load(str(DETECTOR_CKPT), encoder=encoder, device=DEVICE).eval()

vis_centroids_norm, _, _, _ = _pseudo_labels_vis(tile['vis_img'], nsig=CLASSICAL_VIS_NSIG, max_sources=1000)
vis_classical_xy = vis_centroids_to_xy(vis_centroids_norm, tile['vis_hw'])

with torch.no_grad():
    neural_pred = detector.predict(
        tile['images'],
        tile['rms'],
        conf_threshold=NEURAL_CONF_THRESHOLD,
        tile_hw=tile['vis_hw'],
    )

neural_vis_xy = neural_pred['positions_px'].detach().cpu().numpy().astype(np.float32)
neural_scores = neural_pred['scores'].detach().cpu().numpy().astype(np.float32)
vis_compare = compare_detection_sets(neural_vis_xy, vis_classical_xy, radius_px=MATCH_RADIUS_VIS_PX)

print(tile['euclid_path'].name)
print(f'VIS classical detections: {len(vis_classical_xy)}')
print(f'Neural 10-band detections: {len(neural_vis_xy)}')
print(f'Neural matched to VIS classical: {vis_compare["matched_pred_count"]} / {len(neural_vis_xy)}')
print(f'VIS classical recovered by neural: {vis_compare["matched_ref_count"]} / {len(vis_classical_xy)}')
print(f'Median neural score: {np.median(neural_scores):.3f}' if len(neural_scores) else 'Median neural score: n/a')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)

ax = axes[0]
ax.imshow(stretch_image(tile['vis_img']), cmap='gray', origin='lower')
if len(vis_classical_xy):
    ax.scatter(
        vis_classical_xy[:, 0],
        vis_classical_xy[:, 1],
        s=12,
        facecolors='none',
        edgecolors='cyan',
        linewidths=0.7,
    )
ax.set_title(f'VIS classical only | N={len(vis_classical_xy)}')
ax.set_xlim(0, tile['vis_hw'][1])
ax.set_ylim(0, tile['vis_hw'][0])
ax.set_xlabel('x [px]')
ax.set_ylabel('y [px]')

ax = axes[1]
ax.imshow(stretch_image(tile['vis_img']), cmap='gray', origin='lower')
if len(vis_classical_xy):
    vis_only = ~vis_compare['matched_ref_mask']
    ax.scatter(
        vis_classical_xy[vis_only, 0],
        vis_classical_xy[vis_only, 1],
        s=12,
        facecolors='none',
        edgecolors='cyan',
        linewidths=0.7,
        label='VIS-only classical',
    )
if len(neural_vis_xy):
    neural_only = ~vis_compare['matched_pred_mask']
    ax.scatter(
        neural_vis_xy[neural_only, 0],
        neural_vis_xy[neural_only, 1],
        s=18,
        c='magenta',
        marker='x',
        linewidths=0.8,
        label='Neural-only',
    )
    ax.scatter(
        neural_vis_xy[vis_compare['matched_pred_mask'], 0],
        neural_vis_xy[vis_compare['matched_pred_mask'], 1],
        s=18,
        c='lime',
        marker='x',
        linewidths=0.8,
        label='Neural matched to VIS',
    )
ax.set_title(
    f'VIS classical vs 10-band neural | matched {vis_compare["matched_pred_count"]}/{len(neural_vis_xy)} '
    f'| recovered {vis_compare["matched_ref_count"]}/{len(vis_classical_xy)}'
)
ax.set_xlim(0, tile['vis_hw'][1])
ax.set_ylim(0, tile['vis_hw'][0])
ax.set_xlabel('x [px]')
ax.set_ylabel('y [px]')
ax.legend(loc='upper right', fontsize=9)

plt.show()
